# RPS card game: masked PPO self-play

This notebook trains one shared policy through two-seat self-play, heuristic matches, and a league of frozen older policies. It saves PyTorch checkpoints and exports an ONNX model for later integration with the TypeScript server.

The default reward is the actual match outcome (`+1 / 0 / -1`), not one-battle HP difference.

In [ ]:
REPO_URL = "https://github.com/yodsawit/rps-card-game.git"
BRANCH = "main"
UPDATES = 200
EPISODES_PER_UPDATE = 64
EVAL_EPISODES = 100
USE_GOOGLE_DRIVE = True  # Keeps checkpoints after the Colab runtime disconnects
AUTO_RESUME = True  # Uses latest.pt when one already exists in output_dir
SEED = 20260906

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo_dir = Path("/content/rps-card-game")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "training/requirements-colab.txt"], check=True)
print("Repository ready at", repo_dir)

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "training.test_env", "-v"], check=True)

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_dir = Path("/content/drive/MyDrive/rps-self-play")
else:
    output_dir = Path("/content/rps-self-play")
output_dir.mkdir(parents=True, exist_ok=True)
print("Artifacts will be written to", output_dir)

In [ ]:
command = [
    sys.executable, "-m", "training.self_play",
    "--updates", str(UPDATES),
    "--episodes-per-update", str(EPISODES_PER_UPDATE),
    "--eval-episodes", str(EVAL_EPISODES),
    "--seed", str(SEED),
    "--device", "auto",
    "--output-dir", str(output_dir),
]
resume_path = output_dir / "latest.pt"
if AUTO_RESUME and resume_path.exists():
    command.extend(["--resume", str(resume_path)])
    print("Resuming from", resume_path)
print("Running:", " ".join(command))
subprocess.run(command, check=True)

In [ ]:
import json
import matplotlib.pyplot as plt

metrics = [json.loads(line) for line in (output_dir / "metrics.jsonl").read_text().splitlines()]
evaluated = [item for item in metrics if "evaluation" in item]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot([item["update"] for item in metrics], [item["value_loss"] for item in metrics])
axes[0].set(title="Value loss", xlabel="Update")
axes[1].plot([item["update"] for item in evaluated], [item["evaluation"]["win_rate"] for item in evaluated], marker="o")
axes[1].axhline(0.5, color="gray", linestyle="--")
axes[1].set(title="Win rate vs Python heuristic", xlabel="Update", ylim=(0, 1))
plt.show()
print(json.dumps(evaluated[-1]["evaluation"], indent=2) if evaluated else "No evaluation yet.")

In [ ]:
import numpy as np
import onnxruntime as ort

spec = json.loads((output_dir / "model-spec.json").read_text())
session = ort.InferenceSession(str(output_dir / "rps_policy.onnx"), providers=["CPUExecutionProvider"])
dummy_observation = np.zeros((1, spec["observation_size"]), dtype=np.float32)
dummy_mask = np.ones((1, spec["action_size"]), dtype=np.bool_)
policy_logits, value = session.run(None, {"observation": dummy_observation, "legal_mask": dummy_mask})
print("ONNX outputs:", policy_logits.shape, value.shape)

In [ ]:
import shutil
archive = shutil.make_archive("/content/rps-self-play-artifacts", "zip", output_dir)
print("Created", archive)
from google.colab import files
files.download(archive)

## Promotion rule

Bookmark the public GitHub→Colab URL and open it for every session. Colab then loads the latest notebook from GitHub, and its setup cell clones or pulls the latest `main` branch. Google Drive stores only training artifacts and checkpoints.

Do not replace the server's Advanced bot from this notebook alone. First evaluate the ONNX checkpoint against ARC, current Advanced, and historical learned checkpoints using fixed seeds and both seat orders. The server integration must reproduce this notebook's 90-value observation exactly and retain the existing bot as fallback.